# RetailMind AI — Customer Churn Prediction
## Day 1 & Day 2 Practical Notebook

**Dataset:** Telco Customer Churn (Kaggle)  
**Objective:** Build a baseline churn prediction pipeline, then compare Decision Tree, Random Forest and SVM.

### Before running
1. Keep this notebook inside the project's `notebooks/` folder.
2. Keep the CSV inside `data/` with the filename `WA_Fn-UseC_-Telco-Customer-Churn.csv`.
3. Install dependencies from the project root with `python -m pip install -r requirements.txt`.
4. Run the cells from top to bottom.

This notebook is for exploration and learning. The reusable implementation remains in `src/train.py` and `src/compare_models.py`.


## 0. Imports and setup

In [ ]:
from pathlib import Path
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report
)

# This notebook is in notebooks/, so the dataset is one folder above.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())


## 1. Load the Telco Customer Churn dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place the CSV in the project's data/ folder."
    )

df = pd.read_csv(DATA_PATH)
df.head()


## 2. Understand the dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
print("Missing values before conversion:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nTarget distribution:")
display(df["Churn"].value_counts().to_frame("count"))

print("\nTarget distribution (%):")
display((df["Churn"].value_counts(normalize=True) * 100).round(2).to_frame("percentage"))


### Observation
`TotalCharges` is often read as text because some rows contain blank strings. We convert it to numeric so those blanks become missing values, which the preprocessing pipeline will impute. The target is `Churn`: `Yes` means churn and `No` means not churn.


## 3. Clean target and separate features

In [ ]:
# Convert TotalCharges to numeric; invalid/blank values become NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove the identifier because customerID is not a predictive feature.
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

if y.isna().any():
    raise ValueError("Unexpected values found in the Churn target column.")

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Target values:", sorted(y.unique()))


## 4. Identify numerical and categorical features

In [ ]:
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)


## 5. Create preprocessing pipelines

In [ ]:
# Numerical columns: fill missing values with the median, then scale.
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical columns: fill missing values with the most frequent value,
# then one-hot encode categories not seen during training safely.
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created.")


## 6. Create one reproducible train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())
print("\nTesting target proportions:")
display(y_test.value_counts(normalize=True).rename("proportion").to_frame())


# Day 1 — Logistic Regression baseline

## 7. Build and train the baseline pipeline

The preprocessing and classifier are combined into one scikit-learn pipeline. This ensures that preprocessing is fitted using the training data only and then applied consistently to the test data.


In [ ]:
day1_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

day1_model.fit(X_train, y_train)
print("Day 1 Logistic Regression training completed.")


## 8. Evaluate the Day 1 baseline

In [ ]:
day1_pred = day1_model.predict(X_test)
day1_prob = day1_model.predict_proba(X_test)[:, 1]

day1_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, day1_pred),
    "Precision": precision_score(y_test, day1_pred, zero_division=0),
    "Recall": recall_score(y_test, day1_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, day1_prob)
}

day1_results = pd.DataFrame([day1_metrics])
display(day1_results.round(4))

print(classification_report(y_test, day1_pred, target_names=["No churn", "Churn"], zero_division=0))


## 9. Save the Day 1 model and metrics

In [ ]:
models_dir = PROJECT_ROOT / "models"
reports_dir = PROJECT_ROOT / "reports"
models_dir.mkdir(exist_ok=True)
reports_dir.mkdir(exist_ok=True)

joblib.dump(day1_model, models_dir / "churn_pipeline.pkl")
day1_results.to_csv(reports_dir / "baseline_metrics.csv", index=False)

print("Saved:", models_dir / "churn_pipeline.pkl")
print("Saved:", reports_dir / "baseline_metrics.csv")


### Day 1 notes
After running the evaluation cell, record what the baseline metrics tell you. In particular, recall measures the proportion of actual churners correctly identified. For this business problem, missing customers who are likely to churn can be costly.


# Day 2 — Decision Tree, Random Forest and SVM

## 10. Define the three candidate models

We reuse the same `X_train`, `X_test`, `y_train`, `y_test` split and the same preprocessing design from Day 1. This makes the comparison fair.


In [ ]:
candidate_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

print("Models to compare:", list(candidate_models.keys()))


## 11. Train and evaluate all three models

In [ ]:
day2_pipelines = {}
day2_rows = []

for model_name, classifier in candidate_models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    print(f"Training {model_name}...")
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    day2_rows.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

    day2_pipelines[model_name] = pipeline
    print(f"{model_name} completed.")

day2_results = pd.DataFrame(day2_rows)
display(day2_results.round(4))


## 12. Rank models and select the champion

In [ ]:
# Primary selection criterion: highest churn-class recall.
# Tie-breaker: highest ROC-AUC.
ranked_results = day2_results.sort_values(
    by=["Recall", "ROC-AUC"],
    ascending=False
).reset_index(drop=True)

display(ranked_results.round(4))

champion_name = ranked_results.loc[0, "Model"]
champion_model = day2_pipelines[champion_name]

print("Selected Day 2 champion:", champion_name)
print("Selection rule: highest recall, then highest ROC-AUC.")


## 13. Save the Day 2 comparison and champion model

In [ ]:
day2_results_path = reports_dir / "day2_comparison.csv"
champion_path = models_dir / "day2_champion.pkl"

ranked_results.to_csv(day2_results_path, index=False)
joblib.dump(champion_model, champion_path)

print("Saved comparison:", day2_results_path)
print("Saved champion:", champion_path)


## 14. Compare Day 1 baseline with the Day 2 champion

In [ ]:
day1_row = day1_results.copy()
day1_row["Selection"] = "Day 1 baseline"

champion_row = ranked_results.iloc[[0]].copy()
champion_row["Selection"] = "Day 2 champion"

baseline_vs_champion = pd.concat(
    [day1_row, champion_row],
    ignore_index=True
)[["Selection", "Model", "Accuracy", "Precision", "Recall", "ROC-AUC"]]

display(baseline_vs_champion.round(4))


## 15. Final observations and conclusion

After running all cells, complete these notes using your actual results:

- **Day 1 baseline model:** Logistic Regression
- **Day 2 champion model:** Write the model name printed in Step 12.
- **Why was it selected?** It achieved the highest recall on the same held-out test set (ROC-AUC breaks a recall tie).
- **What does recall mean here?** The fraction of actual churners correctly identified.
- **Next improvement:** Consider tuning model hyperparameters and evaluating whether the recall improvement is stable.

Do not fill in or guess metric values before running the notebook.


## Files created by this notebook

- `models/churn_pipeline.pkl` — Day 1 Logistic Regression pipeline
- `reports/baseline_metrics.csv` — Day 1 metrics
- `models/day2_champion.pkl` — Day 2 champion pipeline
- `reports/day2_comparison.csv` — Day 2 model comparison

**GitHub reminder:** Commit the notebook and these generated files if your lab requires them. Do not commit secrets or unrelated environment files.
